In [1]:
from flask import Flask, render_template, request, redirect, url_for, jsonify
from datetime import datetime

In [2]:
app = Flask(__name__)
app.jinja_env.autoescape = False
app

<Flask '__main__'>

각  html의 기능을 구현

In [ ]:
stored_username = "123"
stored_password = "123"
stored_date = "2024-10-11"
logInState = 0

logs = []

@app.route('/')
def index():
    global logInState
    # 홈 페이지를 렌더링하고 로그인 상태를 전달
    return render_template('index.html', logInState=logInState)


@app.route('/sign-in', methods=['GET', 'POST'])
def sign_in():
    return render_template('sign-in.html')

@app.route('/sign-up', methods=['GET', 'POST'])
def sign_up():
    global stored_username, stored_password, stored_date
    if request.method == 'POST':
        # 회원가입 폼에서 사용자 이름과 비밀번호를 가져옴
        new_username = request.form.get('username')
        new_password = request.form.get('password')
        confirm_password = request.form.get('confirm_password')

        # 비밀번호와 비밀번호 확인이 일치하는지 확인
        if new_password == confirm_password:
            stored_username = new_username
            stored_password = new_password
            
            # 현재 날짜를 저장
            stored_date = datetime.now().strftime("%Y-%m-%d")
            
            return redirect(url_for('sign_in'))
        else:
            # 비밀번호가 일치하지 않을 경우 오류 메시지 표시
            return render_template('sign-up.html', error="Passwords do not match. Please try again.")
    return render_template('sign-up.html')



@app.route('/login', methods=['POST'])
def login():
    global logInState
    # 로그인 폼에서 사용자 이름과 비밀번호를 가져옴
    username = request.form.get('username')
    password = request.form.get('password')

    # 자격 증명 검증
    if username == stored_username and password == stored_password:
        logInState = 1  # 로그인 성공 시 로그인 상태를 1로 설정
        return redirect(url_for('products'))
    else:
        # 자격 증명이 유효하지 않은 경우 로그인 페이지에 오류 메시지 표시
        return render_template('sign-in.html', error="Invalid credentials. Please try again.")
    

@app.route('/company')
def company():
    global logInState
    return render_template('company.html', logInState=logInState)

@app.route('/contact')
def contact():
    global logInState
    return render_template('contact.html', logInState=logInState)

@app.route('/products')
def products():
    global logInState
    return render_template('products.html', logInState=logInState)

def get_all_products():
    return {
        1: {'name': 'BREAK OUT & SPACE INVADER', 'description': 'Break bricks or shoot aliens to win!'},
        2: {'name': 'Card Game', 'description': 'Challenge the robot to a card-matching battle!'},
        3: {'name': 'Friday Night Funkin', 'description': 'Press button through Rhythm'},
        4: {'name': "Heaven's says", 'description': 'Prove your faithfulness according to the will of the divine'},
        5: {'name': 'Tetris', 'description': 'Stack and clear lines with falling blocks!'},
        6: {'name': 'Escape in Error', 'description': 'Escape a world full of errors!'}
    }


@app.route('/product_detail/<int:product_id>/')
def product_detail(product_id):
    global logInState
    # 모든 데이터 셋 가져오기
    product = get_all_products().get(product_id)
    products = get_all_products()

    # id에 따라 추가 게임 추천 요소들 고르기
    if product_id in [1, 2, 3]:
        related_products = {k: v for k, v in products.items() if k in [1, 2, 3]}
    elif product_id in [4, 5, 6]:
        related_products = {k: v for k, v in products.items() if k in [4, 5, 6]}
    else:
        related_products = {}

    return render_template(
        'product-detail.html', 
        product_id=product_id, 
        product_name=product['name'], 
        product_description=product['description'], 
        products=related_products, 
        logInState=logInState
    )

@app.route('/save-log', methods=['POST'])
def save_log():
    global logs
    data = request.json
    score = data.get('score')
    product_name = data.get('product_name')
    date = data.get('date')

    # 로그 리스트에 추가
    logs.append({'date': date, 'product_name': product_name, 'score': score})
    return jsonify({'status': 'success'}), 200

@app.route('/user-info', methods=['GET', 'POST'])
def user_info():
    global stored_username, stored_password, logInState, stored_date

    if request.method == 'POST':
        if 'update' in request.form:  # 업데이트 버튼이 클릭되었는지 확인
            # 폼에서 업데이트된 사용자 이름과 비밀번호를 가져옴
            new_username = request.form.get('username')
            new_password = request.form.get('userpassword')

            # 저장된 사용자 이름과 비밀번호를 새로운 값으로 업데이트
            if new_username and new_password:
                stored_username = new_username
                stored_password = new_password
                print("변경이 완료되었습니다.")
    
    return render_template('user-info.html', username=stored_username, userpassword=stored_password, userRegisterDate=stored_date)

@app.route('/logout', methods=['POST'])
def logout():
    global logInState
    logInState = 0  # 로그아웃 시 로그인 상태를 0으로 설정
    print("로그아웃되었습니다")
    return redirect(url_for('index'))


In [4]:
if __name__ == '__main__':
    app.run(debug=True, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [01/Dec/2024 19:39:54] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [01/Dec/2024 19:39:54] "GET /static/css/mainTemplateCss.css HTTP/1.1" 304 -
127.0.0.1 - - [01/Dec/2024 19:39:54] "GET /static/css/bootstrap-icons.css HTTP/1.1" 304 -
127.0.0.1 - - [01/Dec/2024 19:39:54] "GET /static/images/header/LogInBackgroundImg1.jpg HTTP/1.1" 304 -
127.0.0.1 - - [01/Dec/2024 19:39:54] "GET /static/css/bootstrap.min.css HTTP/1.1" 304 -
127.0.0.1 - - [01/Dec/2024 19:39:54] "GET /static/fonts/bootstrap-icons.woff2?856008caa5eb66df68595e734e59580d HTTP/1.1" 304 -
127.0.0.1 - - [01/Dec/2024 19:39:54] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [01/Dec/2024 19:39:58] "GET /static/images/header/LogInBackgroundImg2.jpg HTTP/1.1" 304 -
127.0.0.1 - - [01/Dec/2024 19:40:01] "GET /static/images/header/LogInBackgroundImg3.jpg HTTP/1.1" 304 -
127.0.0.1 - - [01/Dec/2024 19:40:25] "GET /static/images/header/LogInBackgroundImg2.jpg HTTP/1.1" 304 -
1